# ADNI: only the 175 missing runs — no augmentation

Use **filippofocaccia2@gmail.com** for both Drive mounting and Google authentication.
Code, dataset and four recovery TARs have been copied into that account's
`My Drive/hippo_adni_3fold_3seed`. This is ADNI, not MSD.

The fixed allowlist contains **4 interrupted runs to resume + 3 untouched seed-1 runs + 168 seed-4 runs**.
The 329 completed target runs are excluded from scheduling and archive scans. Their saved comparison
snapshot is included for context and labelled as historical, not freshly reverified. Original results
remain untouched in the source account. No seed-2 runs or new experiments/benchmarks are added.

The existing 5% subject splits, folds 0/1/2, seeds 0/1/4, initialization, BF16, batch size,
losses, calibration, optimizers, schedules and 31,976,450,667,090 optimization-FLOP cap are unchanged.
nnUNet keeps native Dice+CE/SGD and 60% constraint targets; the other models keep Dice/AdamW and 10%.
**No augmentation.** Seed replacement after observing results remains exploratory/post-selection.

Use the previously validated **A100-SXM4-80GB** and the saved software versions. Source, environment,
preflight, checkpoint hashes and FLOP ledgers remain mandatory checks. Missing or stale recovery stops
the affected continuation; it cannot silently restart. Run this notebook from the top in the new account.
Do not run the old account's notebook in parallel: the accounts cannot share controller locks.

The v3 speed and Drive improvements are retained: local SSD data/caches/status, adaptive worker slots,
validated acceleration, lossless recovery compression, duplicate-transfer suppression, paced writes,
serialized bulk transfers, bounded quota retries and comparison uploads at most once per five minutes.
Every update still has its durable FLOP charge/readback. This reduces traffic; it cannot guarantee that
Google will never throttle the account. Never repeatedly interrupt a recovery upload.


## 1. Mount Drive and stage the pinned code; preserve existing work


In [ ]:
from pathlib import Path
from contextlib import contextmanager, ExitStack
import fcntl, hashlib, json, os, shutil, signal, subprocess, sys, threading, time, zipfile
@contextmanager
def phase(label):
    print('PHASE:', label, flush=True)
    stop=threading.Event(); started=time.monotonic()
    def pulse():
        while not stop.wait(30):
            value=dict(phase=label, elapsed=time.monotonic()-started, status='still waiting; heartbeat is not progress')
            Path('/content/adni_seeds014_setup_status.json').write_text(json.dumps(value))
            print('HEARTBEAT:', value, flush=True)
    thread=threading.Thread(target=pulse,daemon=True);thread.start()
    try: yield
    finally: stop.set();thread.join()
from google.colab import drive
with phase('mount Drive'): drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/hippo_adni_3fold_3seed')
INPUT=DRIVE_ROOT/'input'
ORIGINAL_RESULTS=DRIVE_ROOT/'results_noaug_v1'
RESULTS=DRIVE_ROOT/'results_seeds014_noaug_v1'
ORIGINAL_BUNDLE=Path('/content/hippo_adni_bundle')
BUNDLE=Path('/content/hippo_adni_seed4_bundle')
PACKAGE=Path('/content/hippo_adni_missing175_v4')
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()
def stage_zip(name, expected, destination):
    local=Path('/content')/name
    with phase('copy '+name+' from Drive to local SSD'):
        if not local.is_file() or sha256(local)!=expected:
            if name=='adni_missing175_noaug_v4_code.zip' and not (INPUT/name).is_file():
                temporary=local.with_suffix(local.suffix+'.partial')
                with temporary.open('wb') as joined_zip:
                    for suffix in ['.part01', '.part02']:
                        with (INPUT/(name+suffix)).open('rb') as chunk:
                            shutil.copyfileobj(chunk,joined_zip,8*1024**2)
                assert sha256(temporary)==expected, 'Incomplete/corrupt code transfer; training remains blocked.'
                temporary.replace(local)
            else:
                shutil.copyfile(INPUT/name,local)
    assert sha256(local)==expected, 'ZIP hash mismatch: '+name
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None
        for i in z.infolist():
            p=Path(i.filename)
            assert not p.is_absolute() and '..' not in p.parts and ((i.external_attr>>16)&0o170000)!=0o120000
        z.extractall(destination)
def verify_files(root, manifest):
    for name, expected in json.loads(manifest.read_text()).items():
        assert sha256(root/name)==expected, 'Changed file: '+name
with phase('confirm this is the destination Drive mount'):
    assert json.loads((DRIVE_ROOT/'destination_account.json').read_text()) == {
        'account':'filippofocaccia2@gmail.com',
        'campaign_folder_id':'1cEfUypMBV9Ay9hNW5nE1_XTJfnGVR9Dm',
        'continuation':'adni_missing175_20261018'}, 'Mount filippofocaccia2@gmail.com; source account must not be used.'
    for folder in (ORIGINAL_RESULTS,RESULTS):
        for name in ('.adni_controller_lock','.adni_seeds014_controller_lock'):
            assert not (folder/name).exists(), 'Existing controller ownership: preserve and inspect it.'
with ExitStack() as locks:
    for root in (ORIGINAL_BUNDLE,BUNDLE):
        (root/'work').mkdir(parents=True,exist_ok=True)
        lock=locks.enter_context((root/'work/sweep.lock').open('a'))
        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    for cmdline in Path('/proc').glob('[0-9]*/cmdline'):
        try: words=cmdline.read_bytes().decode(errors='replace').split('\0')
        except (FileNotFoundError,PermissionError,ProcessLookupError):continue
        for root in (ORIGINAL_BUNDLE,BUNDLE):
            assert not (str(root/'entry.py') in words or (str(root) in words and any(w.endswith(('/worker.py','/phase.py')) for w in words))), 'An ADNI worker is still active.'
    stage_zip('adni_missing175_noaug_v4_code.zip','3cad3a5a26fd4682d4e34a133e28cc05bf33b9a05679f2e3aaf77e9232a2bf55',PACKAGE)
    verify_files(PACKAGE,PACKAGE/'package_manifest.json')
    with phase('stage frozen science and dataset on local SSD'):
        for root, supplied, expected in (
            (ORIGINAL_BUNDLE,PACKAGE/'original_bundle','e0b87c00231bb355aa66531f6684c361a632b3f8e393f6afbfb4939a37d0caeb'),
            (BUNDLE,PACKAGE/'seed4_bundle','88349a82b03b4f93884def610f5ef12077968ef3b1b75192a3472f395266a148'),
        ):
            marker=root/'bundle_manifest.json'
            if marker.exists():
                assert sha256(marker)==expected, 'Preserve changed source; do not overwrite it.'
            else:
                assert not any((root/'work').glob('*/budget_ledger.json')), 'Existing work without manifest requires inspection.'
                shutil.copytree(supplied,root,dirs_exist_ok=True)
            if root==ORIGINAL_BUNDLE:
                data=root/'data/adni_hippocampus_full.pkl'
                if not data.exists():
                    stage_zip('adni_data_verified.zip','e05bb92b0a6f5024dc9c823283a63020dba8f9cbef70141949a854f6bad6e709',root)
            else:
                (root/'data').mkdir(exist_ok=True)
                data=root/'data/adni_hippocampus_full.pkl'
                if not data.exists():data.symlink_to(ORIGINAL_BUNDLE/'data/adni_hippocampus_full.pkl')
            verify_files(root,marker)
print('Continuation results:',ORIGINAL_RESULTS,RESULTS)
print('Historical result archives will not be scanned or copied.')


## 2. Authenticate quota checks and prepare the same training environment


In [ ]:
from google.colab import auth
with phase('authenticate destination account and quota API'):
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    credentials,_=google.auth.default()
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
    account=service.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
    assert account=='filippofocaccia2@gmail.com', 'Authenticate the destination account.'
VENV=Path('/content/hippo_adni_venv');PYTHON=VENV/'bin/python'
def run_logged(args, name):
    path=Path('/content')/('adni_seeds014_'+name+'.log')
    with path.open('a') as log:
        p=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
        def drain():
            for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        interrupted=False
        try:drain();code=p.wait()
        except KeyboardInterrupt:
            interrupted=True
            print('Requesting clean stop; wait for all verified recovery uploads.',flush=True)
            if name in ('run','report'):p.send_signal(signal.SIGINT)
            else:os.killpg(p.pid,signal.SIGINT)
            drain();code=p.wait()
        finally:p.stdout.close()
        if interrupted:raise KeyboardInterrupt('Stopped cleanly; rerun later to resume.')
        if code:raise RuntimeError(f'Exit {code}; inspect {path}. Recovery preserved.')
with phase('prepare training environment'):
    if not PYTHON.exists():run_logged([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup')
    if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
        import urllib.request
        bootstrap=Path('/content/hippo_adni_get_pip.py')
        urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',bootstrap)
        run_logged([PYTHON,bootstrap],'setup')
    run_logged([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',ORIGINAL_BUNDLE/'requirements-colab.txt'],'setup')
COMMON=['--parent-bundle',ORIGINAL_BUNDLE,'--parent-results',ORIGINAL_RESULTS,'--bundle',BUNDLE,'--durable',RESULTS]
def campaign(command,*args):
    run_logged([PYTHON,'-u',PACKAGE/'campaign.py',command,*COMMON,*args],command)
subprocess.run(['nvidia-smi'],check=True)

with phase('import gate receipts and recovery pointers once; never reset accounting'):
    run_logged([PYTHON,PACKAGE/'bootstrap_missing.py','--drive-root',DRIVE_ROOT],'bootstrap')


## 3. Required compatibility preflight (CPU only; no extra experiments)

Verify the unchanged source, saved GPU/software bindings, original passed GPU preflights and
A100 acceleration evidence. Then check Swin initializations for seeds 0/1/4 on CPU: the original
strict tolerance, all RNG checks and exact complete model hashes must pass. The same initialization
hashes are required in all three folds. This does not run forward/backward passes, optimizer updates
or benchmarks, and does not replace the original preflight receipts. A failed check blocks training.
Use the saved A100-SXM4-80GB environment; a different GPU/software environment is not silently accepted.


In [ ]:
with phase('verify saved preflights and CPU Swin initialization; no training trials'):
    for root, durable, control in (
        (ORIGINAL_BUNDLE, ORIGINAL_RESULTS, PACKAGE/'control01'),
        (BUNDLE, RESULTS, PACKAGE/'control04'),
    ):
        run_logged([PYTHON, '-u', PACKAGE/'preflight_initfix.py',
                    '--bundle', root, '--durable', durable, '--control', control], 'preflight')


## 4. Resume four interrupted runs, then finish the allowlist

Only results produced by these 175 tasks are checked and skipped on subsequent sessions.
The original 329 completed archives are never opened. Full checkpoint hash/readability/charge checks
remain mandatory for the four resumed runs. They have priority over untouched work.

Live output and SSD status show phases, workers, updates, GPU activity and Drive waits every 30 seconds.
Worker count is resource-adaptive; GPU RAM occupancy is not GPU utilization. No profiling trials are added.
Use one controller only. Existing STOP/ownership files are respected. Interrupt once, wait for verified
recovery uploads, then rerun this notebook to continue. The 8-hour default is a clean-stop target, not a
Colab runtime guarantee. Hard runtime deletion before upload can still require a recovery audit.


In [ ]:
MAX_NEW_RUNS=175  # Fixed missing-only scope; verified continuation results are skipped.
MAX_WORKERS=0    # Existing CPU/resource-supported adaptive scheduler.
SESSION_HOURS=8.0
campaign('run','--max-runs',MAX_NEW_RUNS,'--workers',MAX_WORKERS,'--hours',SESSION_HOURS)


## 5. Results: saved historical snapshot + verified continuation

This cell starts no training. It checks only continuation archives, never old-account archives.
The 329 historical rows retain their saved metrics and are labelled `historical_saved_snapshot_not_rescanned`.
New rows are labelled `verified_in_continuation`. Gains use matched model/fold/seed baselines.
This reporting provenance must stay visible; the notebook does not claim a fresh audit of the old study.


In [ ]:
campaign('report','--hours',2)
import pandas as pd
result=json.loads((RESULTS/'comparison.json').read_text())
print('Historical saved (not rescanned):',result['historical_saved_count'])
print('Verified continuation:',result['verified_continuation_count'],'/175')
display(pd.DataFrame(result['aggregate']))
display(pd.DataFrame(result['runs']))


Files needed on the destination account are already staged in
`My Drive/hippo_adni_3fold_3seed`: this notebook's matching v4 code ZIP and `adni_data_verified.zip`
in `input/`, plus the four copied TARs under `results_noaug_v1/recovery/`.
Do not replace or remove recovery files, charge ledgers, results, STOP files or ownership markers.
The notebook restores only missing immutable setup metadata on its first launch; later launches never
reimport old checkpoint pointers over newer progress. Original source-account results are preserved.

Local logs: `/content/adni_seeds014_*.log`; live status: each bundle's `parallel_status/`;
Drive counters: `/content/hippo_adni_drive_io/`. Full checkpoint verification happens before resumed
updates. Scheduling/import/accounting failure handling were tested locally; this release adds no GPU
benchmark or scientific run and has not been exercised end-to-end on Colab.

The code ZIP is stored on Drive in two upload parts to fit the connector limit. Setup automatically joins them on SSD and verifies the full ZIP SHA256; no manual assembly is needed.
